# 1. Data processing
**Dataset:** UCI Bike Sharing Dataset (Capital Bikeshare, Washington DC, 2011–2012)

Loads, cleans and engineers features, then saves the results to `processed/` for the other notebooks. **Run this notebook first.**

**Setup (run once in a terminal):**
`pip install ucimlrepo pandas numpy matplotlib scikit-learn`

In [1]:
import os
import numpy as np
import pandas as pd

os.makedirs("processed", exist_ok=True)

## 1. Dataset Description — loading the data
Downloads from the UCI repository. If that fails (no internet), download `hour.csv` manually from
https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset and put it in the same folder as this notebook.

In [2]:
try:
    from ucimlrepo import fetch_ucirepo
    df = fetch_ucirepo(id=275).data.original
    print("Loaded from UCI")
except Exception as e:
    print("UCI download failed, using local hour.csv:", e)
    df = pd.read_csv("hour.csv")

df.to_csv("hour_raw_copy.csv", index=False)   # keep a copy for reproducibility
print(df.shape)
df.head()

Loaded from UCI
(17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [3]:
df.info()
print("\nMissing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  str    
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), str(1)
memory usage: 2.3 MB

Missing values per column:
 instant       0
dteday        0
sea

,count,mean,std,min,25%,50%,75%,max
instant,17379.0,8690.000000,5017.029500,1.00,4345.5000,8690.0000,13034.5000,17379.0000
season,17379.0,2.501640,1.106918,1.00,2.0000,3.0000,3.0000,4.0000
yr,17379.0,0.502561,0.500008,0.00,0.0000,1.0000,1.0000,1.0000
mnth,17379.0,6.537775,3.438776,1.00,4.0000,7.0000,10.0000,12.0000
hr,17379.0,11.546752,6.914405,0.00,6.0000,12.0000,18.0000,23.0000
holiday,17379.0,0.028770,0.167165,0.00,0.0000,0.0000,0.0000,1.0000
weekday,17379.0,3.003683,2.005771,0.00,1.0000,3.0000,5.0000,6.0000
workingday,17379.0,0.682721,0.465431,0.00,0.0000,1.0000,1.0000,1.0000
weathersit,17379.0,1.425283,0.639357,1.00,1.0000,1.0000,2.0000,4.0000
temp,17379.0,0.496987,0.192556,0.02,0.3400,0.5000,0.6600,1.0000


## 2. Data preprocessing
- **Drop `casual` and `registered`** — they add up exactly to `cnt` (the target), so keeping them is data leakage.
- Drop `instant` (just a row number).
- Convert the 0–1 scaled weather values back to real units (scaling factors from the UCI documentation).
- Build a proper datetime column for time-based splitting and plotting.

In [4]:
# Proof of the leakage problem: casual + registered == cnt on every row
import pandas as pd
print("casual + registered == cnt on every row:", (df["casual"] + df["registered"] == df["cnt"]).all())

data = df.drop(columns=["instant", "casual", "registered"]).copy()

data["dteday"] = pd.to_datetime(data["dteday"])
data["datetime"] = data["dteday"] + pd.to_timedelta(data["hr"], unit="h")
data = data.sort_values("datetime").reset_index(drop=True)

# Un-normalise (UCI docs: temp/41, atemp/50, hum/100, windspeed/67)
data["temp_c"] = data["temp"] * 41
data["atemp_c"] = data["atemp"] * 50
data["humidity"] = data["hum"] * 100
data["wind_kmh"] = data["windspeed"] * 67

weather_labels = {1: "Clear", 2: "Mist/cloudy", 3: "Light rain/snow", 4: "Heavy rain"}
season_labels = {1: "Spring", 2: "Summer", 3: "Autumn", 4: "Winter"}
data["weather_name"] = data["weathersit"].map(weather_labels)
data["season_name"] = data["season"].map(season_labels)

print("Weather category counts:\n", data["weathersit"].value_counts().sort_index())
data.head()

casual + registered == cnt on every row: True
Weather category counts:
 weathersit
1    11413
2     4544
3     1419
4        3
Name: count, dtype: int64


,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,...,hum,windspeed,cnt,datetime,temp_c,atemp_c,humidity,wind_kmh,weather_name,season_name
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,...,0.81,0.0,16,2011-01-01 00:00:00,9.84,14.395,81.0,0.0,Clear,Spring
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,...,0.80,0.0,40,2011-01-01 01:00:00,9.02,13.635,80.0,0.0,Clear,Spring
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,...,0.80,0.0,32,2011-01-01 02:00:00,9.02,13.635,80.0,0.0,Clear,Spring
3,2011-01-01,1,0,1,3,0,6,0,1,0.24,...,0.75,0.0,13,2011-01-01 03:00:00,9.84,14.395,75.0,0.0,Clear,Spring
4,2011-01-01,1,0,1,4,0,6,0,1,0.24,...,0.75,0.0,1,2011-01-01 04:00:00,9.84,14.395,75.0,0.0,Clear,Spring


## 3. Feature engineering
- **Cyclical hour** (sin/cos) so the model knows 23:00 and 00:00 are neighbours.
- **Rush-hour flag** for commuter peaks on working days.
- **Hour × working day interaction.**
- **Temperature squared** to capture the "too hot" dip.
- **Log target**, converted back to real bike counts for evaluation.

In [5]:
fe = data.copy()
fe["hr_sin"] = np.sin(2 * np.pi * fe["hr"] / 24)
fe["hr_cos"] = np.cos(2 * np.pi * fe["hr"] / 24)
fe["rush_hour"] = (fe["workingday"].eq(1) & fe["hr"].isin([7, 8, 9, 16, 17, 18, 19])).astype(int)
fe["hr_x_workingday"] = fe["hr"] * fe["workingday"]
fe["temp_sq"] = fe["temp_c"] ** 2
fe["log_cnt"] = np.log1p(fe["cnt"])

# One-hot encode categorical features (they are categories, not quantities)
fe = pd.get_dummies(fe, columns=["weathersit", "season"], prefix=["weather", "season"], drop_first=True, dtype=int)

feature_cols = (
    ["hr", "hr_sin", "hr_cos", "rush_hour", "hr_x_workingday", "workingday", "holiday", "weekday",
     "temp_c", "temp_sq", "humidity", "wind_kmh", "yr", "mnth"]
    + [c for c in fe.columns if c.startswith(("weather_", "season_")) and c not in ("weather_name", "season_name")]
)
# atemp_c deliberately excluded: almost identical to temp_c (see correlation matrix)
print(len(feature_cols), "features:", feature_cols)

20 features: ['hr', 'hr_sin', 'hr_cos', 'rush_hour', 'hr_x_workingday', 'workingday', 'holiday', 'weekday', 'temp_c', 'temp_sq', 'humidity', 'wind_kmh', 'yr', 'mnth', 'weather_2', 'weather_3', 'weather_4', 'season_2', 'season_3', 'season_4']


## Save for the other notebooks

In [6]:
data.to_pickle("processed/data.pkl")
fe.to_pickle("processed/features.pkl")
pd.Series(feature_cols).to_json("processed/feature_cols.json", orient="values")
print("Saved data, features and feature list to processed/")

Saved data, features and feature list to processed/